# Mini CRSI: harness self-improvement that continues, with a memory

A small, runnable version of the paper *Self-Improvement Memory Enables Continual Recursive
Self-Improvement* (arXiv 2611.18856), in the coding domain.

**What happens in this notebook**

1. A small model (the **policy**) solves coding tasks through a **harness**: one Python file that
   builds its prompts, runs its shell commands and decides when it is done.
2. A stronger model (the **engineer**) reads the failures and rewrites the harness. Every rewrite
   that is kept becomes a new version file: `v000.py`, `v001.py`, ...
3. The engineer keeps a logbook, the **Self-Improvement Memory (SIM)**: which problem each edit
   addressed, which tasks it was predicted to fix, and what really happened.
4. Then the tasks change (source benchmark to target benchmark) and optimization continues in four
   ways: from scratch, with the evolved harness only, with problem names only, and with the full
   memory. Comparing them is the paper's experiment.

| Paper | Here |
|---|---|
| Terminal-Bench 2.1 to 4, in containers | 12 source + 12 target tasks, each in a Docker container |
| Policy model, frozen | Claude Haiku through the `claude` CLI |
| Proposer and selector | Claude Sonnet through the `claude` CLI |
| 10 rounds x 5 candidates x 5 rollouts | 4 x 3 x 2 by default (change it in *Settings*) |
| Harness with many files | one Python file per version |

**Before you run it**

- Docker must be running. Every command the agent runs happens in a throwaway container with no
  network.
- Model calls go through the `claude` command-line tool and use your Claude Code subscription.
- Every model reply and every evaluation is cached under `cache/`. Running a cell again is free and
  fast. Changing a setting (model, budget, prompt) makes new calls.
- This is a miniature. With so few tasks, one rollout moves the score by several points, so read the
  numbers as an illustration of the mechanism and not as a reproduction of the paper's results.

In [ ]:
import os, shutil
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "minicrsi").is_dir(), "Start Jupyter from the project folder (the one that contains minicrsi/)."

from minicrsi import report
from minicrsi.evaluate import Budget, evaluate, render_trace
from minicrsi.llm import ClaudeCLI, CodexCLI
from minicrsi.rsi import CONDITIONS, RSIConfig, load_run, run_conditions, run_rsi
from minicrsi.sandbox import docker_available
from minicrsi.sim import SIM
from minicrsi.tasks import load, split

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

assert docker_available(), "Docker is not running."
assert shutil.which("claude"), "The `claude` command-line tool was not found."
print("ready")

## 1. Settings

Everything you are likely to change is in this one cell.

In [ ]:
# Models. "haiku", "sonnet" and "opus" are aliases understood by the `claude` CLI.
POLICY_MODEL, POLICY_EFFORT = "haiku", "low"           # the frozen model that solves the tasks
ENGINEER_MODEL, ENGINEER_EFFORT = "sonnet", "medium"   # the model that edits the harness

# Optimization budget. The paper uses 10 rounds x 5 candidates x 5 rollouts.
ROUNDS, CANDIDATES, ROLLOUTS = 4, 3, 2

# Per-rollout budget, enforced outside the harness so that an edit cannot buy more attempts.
BUDGET = Budget(max_turns=8, max_commands=40, cmd_timeout_cap=30)

SELECTOR = "rule"      # "rule": the best evolve score wins.  "llm": the engineer chooses and reads SIM.
WORKERS = 6            # rollouts run in parallel
SPLIT_SEED = 0         # controls which tasks are evolve and which are held-out
PREFIX = ""            # prefix for run folders under runs/

if os.environ.get("CRSI_SMOKE") == "1":   # tiny settings, only for checking that the notebook runs
    ROUNDS, CANDIDATES, ROLLOUTS, PREFIX = 1, 2, 1, "smoke-"

SEED_HARNESS = ROOT / "harness" / "seed_v000.py"
CACHE, RUNS = ROOT / "cache", ROOT / "runs"

policy = ClaudeCLI(POLICY_MODEL, effort=POLICY_EFFORT, cache_dir=CACHE / "llm")
engineer = ClaudeCLI(ENGINEER_MODEL, effort=ENGINEER_EFFORT, cache_dir=CACHE / "llm")


def config(benchmark, stage, memory_mode="full"):
    return RSIConfig(rounds=ROUNDS, candidates=CANDIDATES, rollouts=ROLLOUTS, budget=BUDGET,
                     memory_mode=memory_mode, selector=SELECTOR, benchmark=benchmark, stage=stage,
                     workers=WORKERS, cache_dir=CACHE)


def score(harness, tasks, label):
    return evaluate(harness, tasks, policy, rollouts=ROLLOUTS, budget=BUDGET, workers=WORKERS,
                    cache_dir=CACHE / "evals", label=label)

## 2. The tasks

There are two benchmarks, both in the coding domain.

- **source**: "make it correct" tasks. The agent implements or fixes a small program so that hidden
  tests pass.
- **target**: every task ships a `check.py` that the agent may run. It confirms only that the answer
  is *valid* (optimisation tasks) or that the *normal case* works (contract tasks). The hidden
  verifier wants more: the best possible answer, or every rule of the specification.

That difference is the point. On the source, "my commands ran and the tests pass" is good evidence
that a task is done. On the target it is not, so a harness tuned on the source needs revising.

Each benchmark is split in half. The engineer only ever sees the **evolve** half. Reported scores
come from the **held-out** half.

One thing measured while building this: with Haiku as the policy, the seed harness already solves
most of the optimisation tasks. They mainly act as a check that an edit does not break what works.
Most of the room to improve on the target is in the contract tasks.

In [ ]:
src_evolve, src_heldout = split(load("source"), seed=SPLIT_SEED)
tgt_evolve, tgt_heldout = split(load("target"), seed=SPLIT_SEED)

rows = [{"benchmark": bench, "split": part, "task": t.name, "kind": t.kind, "instruction": t.instruction}
        for bench, part, tasks in [("source", "evolve", src_evolve), ("source", "held-out", src_heldout),
                                   ("target", "evolve", tgt_evolve), ("target", "held-out", tgt_heldout)]
        for t in tasks]
pd.DataFrame(rows)

In [ ]:
# Look at one task: what the agent is told, what is in its working directory, and the hidden check.
task = next(t for t in load("target") if t.name == "boxes")
print("INSTRUCTION (all the harness receives):\n ", task.instruction)
print("\nFILES IN /work:", list(task.files))
print("\n--- SPEC.md ---\n" + task.files["SPEC.md"])
print("--- hidden verifier (never shown to the agent or the harness) ---" + task.verify[:700] + "...")

In [ ]:
# Optional: prove that every task is sound (the untouched state fails, a reference solution passes).
# Needs no model calls, only Docker. Takes about a minute.
RUN_SELFCHECK = False
if RUN_SELFCHECK:
    from minicrsi import selfcheck
    selfcheck.run()

## 3. The seed harness (version 000)

This is the unevolved starting point. It is deliberately minimal: the model answers in JSON, the
harness runs the commands and shows some of the output. Read it and look for weaknesses before you
let the engineer find them.

In [ ]:
print(SEED_HARNESS.read_text())

In [ ]:
# The contract every harness version must respect. The engineer is given exactly this text.
print(BUDGET.contract())

## 4. How good is the seed?

`evaluate` runs every task `ROLLOUTS` times. The score is pass@1 on a 0 to 100 scale: the share of
rollouts whose hidden verifier passed.

In [ ]:
seed_on_source = score(SEED_HARNESS, src_evolve, "seed on source evolve")
pd.Series(seed_on_source.per_task, name=f"passes out of {ROLLOUTS}").to_frame()

In [ ]:
# One failing rollout, exactly as the engineer will see it. "real stdout / real stderr" is what the
# command produced; "harness sent" is the part the harness passed on to the model.
failed = [r for r in seed_on_source.results if not r.passed]
print(render_trace(failed[0], max_chars=3500) if failed else "No failing rollout in this evaluation.")

## 5. The memory (SIM), by itself

SIM is the engineer's logbook. One entry is one recurring harness problem, with the list of edits
tried for it. Each attempt stores the edit, the reason, the tasks it was **predicted** to fix
(written down before the evaluation), what it **actually** did, and where it happened.

This cell builds a tiny memory by hand, with no model calls, so you can see the two views the
experiment uses.

In [ ]:
demo = SIM()
pid = demo.record(
    problem_id=None, problem_name="the agent says it is done without evidence that the work runs", opened_round=1,
    attempt={"round": 1, "sample": "s2", "what": "refuse 'done' while the last test command failed",
             "why": "rollouts end right after a failing test run", "predicted": ["calc", "ledger"],
             "fixed": ["calc"], "missed": ["ledger"], "regressed": [], "passed_before": 7, "passed": 9,
             "total": 12, "stage": "source", "benchmark": "source", "policy": "claude:haiku"})
demo.record(   # a later attempt that cites the same problem is appended to the same entry
    problem_id=pid, problem_name="", opened_round=3,
    attempt={"round": 3, "sample": "s1", "what": "also require a passing check.py before 'done'",
             "why": "the gate did not look at task checkers", "predicted": ["boxes"], "fixed": [],
             "missed": ["boxes"], "regressed": [], "passed_before": 8, "passed": 8, "total": 12,
             "stage": "target", "benchmark": "target", "policy": "claude:haiku"})

print("=== what the engineer reads with memory_mode='full' (Full SIM) ===\n")
print(demo.render("full"))
print("\n=== what it reads with memory_mode='problems' (Problem-Only Memory) ===\n")
print(demo.render("problems"))

## 6. Experiment A: improve the harness on the source benchmark

One round is:

1. evaluate the current harness on the evolve tasks,
2. the engineer plans `CANDIDATES` different edits and predicts which failing tasks each will fix,
3. the engineer writes a full new harness file per edit,
4. each file is screened (allowed imports only, right interface) and then evaluated,
5. the selector keeps the best candidate if it beats the current harness, otherwise nothing changes,
6. the kept edit and its measured outcome are written into SIM.

The run starts with an empty memory and ends with an evolved harness and a filled memory. Those two
things are what gets carried to the target.

In [ ]:
source = run_rsi(PREFIX + "source", SEED_HARNESS, src_evolve, src_heldout, policy, engineer,
                 config("source", "source"), runs_dir=RUNS)

In [ ]:
# Every candidate of the run: the diagnosed root cause, the edit, the prediction and the result.
report.candidates_table(source)

In [ ]:
# The memory the source run produced. This is what Full SIM carries to the target.
print(source.sim.render("full"))

In [ ]:
# The harness versions that were kept, and what changed from the seed to the final one.
print("versions:", [p.name for p in source.versions], "\n")
report.show_diff(source.versions[0], source.final_harness, max_lines=150)

## 7. Experiment B: the tasks change

Now optimization continues on the **target** benchmark with the same policy model. This is the
paper's cross-dataset setting. Four conditions are compared. All use the same target tasks, the same
policy and the same budget.

| Condition | Starts from | What the engineer may read from SIM |
|---|---|---|
| Cold Start | seed harness | nothing |
| Harness Transfer | source-evolved harness | nothing |
| Problem-Only Memory | source-evolved harness | root-cause names only |
| Full SIM | source-evolved harness | the full record, extended during the target run |

The fair comparison for the memory is **Full SIM against Harness Transfer**: both start from the same
harness file.

Each condition is a full run. Trim `RUN` if you want a shorter experiment.

In [ ]:
RUN = ["cold_start", "harness_transfer", "problem_only", "full_sim"]

target = run_conditions(PREFIX + "target", SEED_HARNESS, source, tgt_evolve, tgt_heldout, policy, engineer,
                        config("target", "target"), conditions=RUN, runs_dir=RUNS)

## 8. Results

*Before* is the starting harness on the held-out target tasks, *after* is the harness kept after the
last round. Note the first number of the transfer conditions: an evolved harness can start below the
seed, which is the observation that motivates the paper.

In [ ]:
report.scores_table(target)

In [ ]:
fig = report.plot_curves(target, split="heldout", title="Held-out target score of the retained harness after each round")
report.curve_table(target, split="heldout")

In [ ]:
# Which edit each run kept in each round, with the SIM problem it was filed under.
# Compare what Full SIM did to the inherited mechanisms with what Harness Transfer did.
report.edit_history(target)

In [ ]:
# How often did "this edit will fix task X" come true?
report.prediction_table(target)

In [ ]:
# The memory after the Full SIM target run: source attempts and target attempts side by side.
key = "full_sim" if "full_sim" in target else list(target)[-1]
print(target[key].sim.render("full"))
report.sim_table(target[key].sim)

In [ ]:
# What a target run changed in the harness it started from. Change `key` to look at another run.
key = "full_sim" if "full_sim" in target else list(target)[-1]
report.show_diff(target[key].versions[0], target[key].final_harness, max_lines=150)

### Reading these numbers honestly

- **The noise is large.** With 6 held-out tasks and 2 rollouts, one rollout is worth 8.3 points. A
  difference of one or two rollouts between conditions is not evidence. The paper needed ten paired
  runs on a much larger benchmark to show an average advantage of about 2.4 points.
- **Each condition is a single run.** To judge a difference, repeat the experiment with other
  `SPLIT_SEED` values or run names, and raise `ROLLOUTS`.
- **What the miniature shows reliably is behaviour**: what the engineer does to inherited mechanisms
  when it can read why they exist, compared with when it cannot. The edit history and the diffs are
  the interesting output.

## 9. Experiments to try

- **Use the engineer as selector**: set `SELECTOR = "llm"`. The selector then reads SIM too, as in
  the paper.
- **More budget**: raise `ROUNDS`, `CANDIDATES` or `ROLLOUTS`. Rollouts reduce noise; rounds give the
  memory more to work with.
- **A harder or easier agent**: change `BUDGET` (for example `max_turns=6`) or `POLICY_EFFORT`.
- **Write a harness yourself**: copy `harness/seed_v000.py`, edit it, and score it with
  `score("harness/my_version.py", tgt_evolve, "mine")`.
- **Change the model instead of the tasks** (the paper's cross-model setting): the next cell.

In [ ]:
# Cross-model CRSI: the benchmark and the split stay the same and the policy model changes.
# A harness is first evolved for one policy, then optimization continues for another policy.
RUN_CROSS_MODEL = False

if RUN_CROSS_MODEL:
    new_policy = ClaudeCLI("sonnet", effort="low", cache_dir=CACHE / "llm")
    # To use the Codex subscription for the second policy instead:
    # new_policy = CodexCLI("default", cache_dir=CACHE / "llm")

    xm_source = run_rsi(PREFIX + "xm-source", SEED_HARNESS, tgt_evolve, tgt_heldout, policy, engineer,
                        config("target", "source"), runs_dir=RUNS)
    xm = run_conditions(PREFIX + "xm", SEED_HARNESS, xm_source, tgt_evolve, tgt_heldout, new_policy, engineer,
                        config("target", "target"), conditions=["harness_transfer", "full_sim"], runs_dir=RUNS)
    display(report.scores_table(xm))

In [ ]:
# A finished run can be reloaded from disk without any model calls, for example:
#   old = load_run("source", RUNS)
# Everything a run produced is in runs/<name>/ : harness/v000.py ..., candidates/, sim.json, rounds.json

print("policy  ", policy.usage.summary())
print("engineer", engineer.usage.summary())